<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#8B0000; overflow:hidden"><b>  Self-Supervised Learning for Tabular Data (SimCLR Style) </b></div>

# 🔬 Self-Supervised Learning for Tabular Data (SimCLR-Style)

## 📌 Motivation
Most machine learning models for tabular data rely heavily on **labeled datasets**, which are often expensive, noisy, or limited in size.  
In contrast, **Self-Supervised Learning (SSL)** enables models to learn meaningful representations **without using labels**, leveraging the structure of the data itself.

While SSL has shown remarkable success in **computer vision** and **natural language processing**, it remains **underexplored for tabular data**despite tabular data being the most common format in real-world applications.

This notebook demonstrates how **contrastive self-supervised learning**, inspired by **SimCLR**, can be adapted to tabular datasets.

---

## 🧠 Key Idea
Instead of predicting labels, the model learns by:
1. Creating **two different augmented views** of the same data point  
2. Training an encoder to **pull similar samples closer** in representation space  
3. Pushing representations of different samples apart  

After self-supervised pretraining, the learned encoder is **frozen** and evaluated on a downstream supervised task using a **linear probe**.

---

## 🏗️ Method Overview
The workflow consists of two main phases:

### 🔹 Phase 1: Self-Supervised Pretraining
- Apply **tabular-specific augmentations**:
  - Gaussian noise injection
  - Feature masking
- Pass augmented samples through a shared encoder network
- Optimize a **contrastive loss (NT-Xent)** to learn representations

### 🔹 Phase 2: Downstream Fine-Tuning
- Freeze the pretrained encoder
- Train a lightweight classifier on labeled data
- Evaluate performance on a test set

---

## 🧪 Dataset
We use a **tabular classification dataset** to illustrate:
- Learning without labels during pretraining
- Performance gains from representation learning

This setup mirrors realistic scenarios where large unlabeled datasets are available, but labeled samples are scarce.

---

## 📊 What You Will Learn
By the end of this notebook, you will understand:
- How self-supervised learning works for tabular data
- How to design augmentations for non-image data
- How contrastive learning improves downstream performance
- Why SSL can outperform purely supervised models in low-label regimes

---



## 🏁 Conclusion
This notebook demonstrates that **self-supervised learning is not limited to images or text**.  
With appropriate augmentations and objectives, SSL can significantly enhance representation learning for tabular data—opening new directions for both research and practical machine learning systems.

⭐ If you find this notebook useful, consider upvoting and sharing!


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Installing Libraries </b></div>

In [1]:


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Loading Data </b></div>

In [2]:
# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Tabular augmentations </b></div>

In [3]:

def augment(x):
    noise = torch.randn_like(x) * 0.05
    mask = torch.rand_like(x) > 0.9
    x_aug = x + noise
    x_aug[mask] = 0
    return x_aug


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Encoder network </b></div>

In [4]:
# Encoder network
class Encoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.net(x)


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Contrastive loss </b></div>

In [5]:
# Contrastive loss
def nt_xent(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    similarity = torch.mm(z1, z2.t()) / temperature
    labels = torch.arange(z1.size(0)).to(z1.device)
    return F.cross_entropy(similarity, labels)


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> SSL training </b></div>

In [6]:
# SSL training
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = Encoder(X.shape[1]).to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

for epoch in range(30):
    x1 = augment(X_tensor)
    x2 = augment(X_tensor)
    z1 = encoder(x1)
    z2 = encoder(x2)
    loss = nt_xent(z1, z2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1} | SSL Loss: {loss.item():.4f}")


Epoch 1 | SSL Loss: 6.1441
Epoch 2 | SSL Loss: 6.1233
Epoch 3 | SSL Loss: 6.1234
Epoch 4 | SSL Loss: 6.1171
Epoch 5 | SSL Loss: 6.1161
Epoch 6 | SSL Loss: 6.1186
Epoch 7 | SSL Loss: 6.1154
Epoch 8 | SSL Loss: 6.1156
Epoch 9 | SSL Loss: 6.1163
Epoch 10 | SSL Loss: 6.1138
Epoch 11 | SSL Loss: 6.1162
Epoch 12 | SSL Loss: 6.1139
Epoch 13 | SSL Loss: 6.1118
Epoch 14 | SSL Loss: 6.1108
Epoch 15 | SSL Loss: 6.1089
Epoch 16 | SSL Loss: 6.1036
Epoch 17 | SSL Loss: 6.1039
Epoch 18 | SSL Loss: 6.1089
Epoch 19 | SSL Loss: 6.0964
Epoch 20 | SSL Loss: 6.0869
Epoch 21 | SSL Loss: 6.0646
Epoch 22 | SSL Loss: 6.0568
Epoch 23 | SSL Loss: 6.0422
Epoch 24 | SSL Loss: 6.0283
Epoch 25 | SSL Loss: 5.9993
Epoch 26 | SSL Loss: 5.9624
Epoch 27 | SSL Loss: 5.8955
Epoch 28 | SSL Loss: 5.8775
Epoch 29 | SSL Loss: 5.8630
Epoch 30 | SSL Loss: 5.8500


In [7]:
# Freeze encoder + linear probe
for p in encoder.parameters():
    p.requires_grad = False

classifier = nn.Linear(128, 1).to(device)
opt = torch.optim.Adam(classifier.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)


In [8]:
for epoch in range(20):
    features = encoder(X_train_t)
    logits = classifier(features).squeeze()
    loss = loss_fn(logits, y_train_t)
    opt.zero_grad()
    loss.backward()
    opt.step()


In [9]:
# Evaluation
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    preds = torch.sigmoid(classifier(encoder(X_test_t))).cpu().numpy()
preds = (preds > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.9298245614035088
